In [3]:
import tkinter as tk
from tkinter import scrolledtext, ttk
import pyperclip
import re
import pandas as pd
import sys
from datetime import timedelta, datetime
from threading import Thread
import keyboard
import pygetwindow as gw
import time
import json
from PIL import Image, ImageTk

In [4]:
import getpass # для определения текущего пользователя позже убрать
env = getpass.getuser()

In [5]:
# 0.1 Куда сохранять эксель
xlsx_path = r"C:\Users\lutzb\Desktop\wt_stats\data.xlsx" if env == 'lutzb' else r"D:\py\wt_stats\data.xlsx"

In [54]:
with pd.ExcelFile(xlsx_path, engine='openpyxl') as xls:
    # Пытаемся прочитать лист 'battles'
    if 'battles' in xls.sheet_names:
        df_for_start_page = pd.read_excel(xls, sheet_name='vehicles', engine='openpyxl')
    else:
        print('ошибка')


In [ ]:
df_for_start_page.tail(10)

In [ ]:
# Добавляем столбец objectives
df_for_start_page['objectives'] = (
    df_for_start_page['kills'] +
    df_for_start_page['kills_air'] +
    df_for_start_page['crits'] +
    df_for_start_page['assists'] +
    df_for_start_page['base_caps']
)
# Добавляем столбец k/d
df_for_start_page['kd'] = (
    (df_for_start_page['kills'] + df_for_start_page['kills_air'] ) /
    (df_for_start_page['did_died'] + df_for_start_page['doubler_used'])
)

In [56]:
# Новый датафрейм через groupby по имени машинки
df_for_start_page = df_for_start_page.groupby('vehicle', as_index=False).agg(
    avg_sl = ('sl_corrected', 'mean'),
    avg_rp = ('rp_corrected', 'mean'),
    avg_mp = ('mp', 'mean'),
    battle_count = ('battle_id', 'count'),
    objectives = ('objectives', 'mean'),
    avg_kd = ('kd', 'mean'),
    total_did_died=('did_died', 'sum'),
    total_doubler_used=('doubler_used', 'sum')
)

# Добавить выживаемость
df_for_start_page['survivability'] = (
    (df_for_start_page['battle_count'] - df_for_start_page['total_did_died']) / df_for_start_page['battle_count']
)

df_for_start_page

,vehicle,avg_sl,avg_rp,avg_mp,battle_count,objectives,avg_kd,total_did_died,total_doubler_used,survivability
0,2С25,4349.105263,790.473684,248.578947,19,1.631579,inf,4.0,0.0,0.789474
1,2С3М,2021.888889,408.333333,134.666667,9,0.444444,inf,1.0,0.0,0.888889
2,AD-4,4185.000000,526.875000,355.625000,8,2.500000,inf,3.0,0.0,0.625000
3,AMC.35 (ACG.1),561.333333,520.333333,506.666667,3,3.000000,inf,0.0,0.0,1.000000
4,AMD.35,380.125000,662.750000,427.500000,8,2.375000,inf,1.0,0.0,0.875000
...,...,...,...,...,...,...,...,...,...,...
143,Т-62М-1,4417.857143,818.571429,273.571429,7,1.857143,inf,2.0,0.0,0.714286
144,Т-72А,4870.894737,865.000000,290.315789,19,1.789474,inf,5.0,0.0,0.736842
145,Як-9УТ,5956.272727,664.636364,490.454545,11,3.454545,inf,1.0,0.0,0.909091
146,◌С-3604,2692.333333,563.333333,409.666667,3,2.333333,inf,0.0,0.0,1.000000


In [57]:
df_for_start_page = df_for_start_page[df_for_start_page['battle_count'] >= 5]

In [58]:
# Получаем топ-3 по avg_sl
top3_sl = df_for_start_page.nlargest(3, 'avg_sl')[['vehicle', 'avg_sl']].values

# Получаем топ-3 по avg_rp
top3_rp = df_for_start_page.nlargest(3, 'avg_rp')[['vehicle', 'avg_rp']].values

# Получаем топ-3 по avg_mp
top3_mp = df_for_start_page.nlargest(3, 'avg_mp')[['vehicle', 'avg_mp']].values

# Топы
# По боям
i = df_for_start_page['battle_count'].idxmax()
name_max_battle_count = df_for_start_page.loc[i, 'vehicle']
value_max_battle_count = df_for_start_page.loc[i, 'battle_count']
# По кд
i = df_for_start_page['avg_kd'].idxmax()
name_max_kd = df_for_start_page.loc[i, 'vehicle']
value_max_kd = df_for_start_page.loc[i, 'avg_kd']
# По objectives
i = df_for_start_page['objectives'].idxmax()
name_max_objectives = df_for_start_page.loc[i, 'vehicle']
value_max_objectives = df_for_start_page.loc[i, 'objectives']
# По survivability
i = df_for_start_page['survivability'].idxmax()
name_max_survivability = df_for_start_page.loc[i, 'vehicle']
value_max_survivability = df_for_start_page.loc[i, 'survivability']

# Выводим в f-строке
print(f"""
Топ 3 по SL:
1. {top3_sl[0][0]} ({round(top3_sl[0][1]):_})
2. {top3_sl[1][0]} ({round(top3_sl[1][1]):_})
3. {top3_sl[2][0]} ({round(top3_sl[2][1]):_})

Топ 3 по RP:
1. {top3_rp[0][0]} ({round(top3_rp[0][1]):_})
2. {top3_rp[1][0]} ({round(top3_rp[1][1]):_})
3. {top3_rp[2][0]} ({round(top3_rp[2][1]):_})

Топ 3 по MP:
1. {top3_mp[0][0]} ({round(top3_mp[0][1]):_})
2. {top3_mp[1][0]} ({round(top3_mp[1][1]):_})
3. {top3_mp[2][0]} ({round(top3_mp[2][1]):_})

Топы по:
Боям - {name_max_battle_count} ({value_max_battle_count} боев)
КД - {name_max_kd} ({value_max_kd})
Полезности - {name_max_objectives} ({round(value_max_objectives)} килов, критов, ассистов, баз)
Выживаемости - {name_max_survivability} ({value_max_survivability})


""".replace("_", " "))


Топ 3 по SL:
1. ИС-6 (29 288)
2. T58 (24 971)
3. Magach 3 (ERA) (22 569)

Топ 3 по RP:
1. Magach 3 (ERA) (4 669)
2. ИС-6 (4 523)
3. T58 (4 313)

Топ 3 по MP:
1. M-51 (1 150)
2. Strv m/41 S-I (938)
3. ИС-6 (821)

Топы по:
Боям - T58 (60 боев)
КД - 2С25 (inf)
Полезности - M-51 (9 килов, критов, ассистов, баз)
Выживаемости - F8F-1B (1.0)





In [ ]:
# Найти максимумы по всем числовым колонкам
numeric_cols = ['avg_sl', 'avg_rp', 'avg_mp']
max_values = df_for_start_page[numeric_cols].max()
print(max_values)

In [ ]:
# Найти индекс строки с максимальным значением
top_sl_name = df_for_start_page['avg_sl'].idxmax()
top_sl_value = round(df_for_start_page.loc[top_sl_name, 'avg_sl'])

top_rp_name = df_for_start_page['avg_rp'].idxmax()
top_rp_value = round(df_for_start_page.loc[top_rp_name, 'avg_rp'])

top_mp_name = df_for_start_page['avg_mp'].idxmax()
top_mp_value = round(df_for_start_page.loc[top_mp_name, 'avg_mp'])


print(f"""
      По SL - {top_sl_name} ({top_sl_value:_})
      По RP - {top_rp_name} ({top_rp_value:_})
      По MP - {top_mp_name} ({top_mp_value:_})
      """.replace("_", " "))

In [ ]:
# Найти индекс строки с максимальным значением
i = df_for_start_page['battle_count'].idxmax()
max_objective_name = df_for_start_page.loc[i, 'vehicle']
max_objective_value = df_for_start_page.loc[i, 'battle_count']
print(max_objective_name, max_objective_value)

In [ ]:
df_for_start_page.query("""
    battle_count >= 5 & 
    avg_sl > 1000 &
    vehicle.str.contains('ИС')
""", engine='python')

In [ ]:
###########

In [59]:
# Функция получения данных для стартовой страницы
def generate_data_for_start_page(xlsx_path):
    # 1. Открываем эксель
    with pd.ExcelFile(xlsx_path, engine='openpyxl') as xls:
    # Пытаемся прочитать лист 'battles'
        if 'battles' in xls.sheet_names:
            df_for_start_page = pd.read_excel(xls, sheet_name='vehicles', engine='openpyxl')
        else:
            print('ошибка чтения стартового xlsx')
    
    # 2. Обрабатываем, готовим df_for_start_page со средними сгруппированный по vehicles

    battle_count = df_for_start_page['battle_id'].nunique()

    # Добавляем столбец objectives
    df_for_start_page['objectives'] = (
        df_for_start_page['kills'] +
        df_for_start_page['kills_air'] +
        df_for_start_page['crits'] +
        df_for_start_page['assists'] +
        df_for_start_page['base_caps']
    )
    # Добавляем столбец k/d
    df_for_start_page['kd'] = (
        (df_for_start_page['kills'] + df_for_start_page['kills_air'] ) /
        (df_for_start_page['did_died'] + df_for_start_page['doubler_used'])
    )

    # Новый датафрейм через groupby по имени машинки
    df_for_start_page = df_for_start_page.groupby('vehicle', as_index=False).agg(
        avg_sl = ('sl_corrected', 'mean'),
        avg_rp = ('rp_corrected', 'mean'),
        avg_mp = ('mp', 'mean'),
        battle_count = ('battle_id', 'count'),
        objectives = ('objectives', 'mean'),
        avg_kd = ('kd', 'mean'),
        total_did_died=('did_died', 'sum'),
        total_doubler_used=('doubler_used', 'sum')
    )

    # Добавляем столбец выживаемость
    df_for_start_page['survivability'] = (
        (df_for_start_page['battle_count'] - df_for_start_page['total_did_died']) / df_for_start_page['battle_count']
    )

    # Убираем технику, у которой менее 5 боев
    df_for_start_page = df_for_start_page[df_for_start_page['battle_count'] >= 5]

    # 3. Рассчитываем нужные показатели
    # Получаем топ-3 по avg_sl
    top3_sl = df_for_start_page.nlargest(3, 'avg_sl')[['vehicle', 'avg_sl']].values

    # Получаем топ-3 по avg_rp
    top3_rp = df_for_start_page.nlargest(3, 'avg_rp')[['vehicle', 'avg_rp']].values

    # Получаем топ-3 по avg_mp
    top3_mp = df_for_start_page.nlargest(3, 'avg_mp')[['vehicle', 'avg_mp']].values

    # Топы
    # По боям
    i = df_for_start_page['battle_count'].idxmax()
    name_max_battle_count = df_for_start_page.loc[i, 'vehicle']
    value_max_battle_count = df_for_start_page.loc[i, 'battle_count']
    # По кд
    i = df_for_start_page['avg_kd'].idxmax()
    name_max_kd = df_for_start_page.loc[i, 'vehicle']
    value_max_kd = df_for_start_page.loc[i, 'avg_kd']
    # По objectives
    i = df_for_start_page['objectives'].idxmax()
    name_max_objectives = df_for_start_page.loc[i, 'vehicle']
    value_max_objectives = df_for_start_page.loc[i, 'objectives']
    # По survivability
    i = df_for_start_page['survivability'].idxmax()
    name_max_survivability = df_for_start_page.loc[i, 'vehicle']
    value_max_survivability = df_for_start_page.loc[i, 'survivability']

    # Выводим в f-строке
    print(f"""
    {battle_count} боев
    Топ 3 по SL:
    1. {top3_sl[0][0]} ({round(top3_sl[0][1]):_})
    2. {top3_sl[1][0]} ({round(top3_sl[1][1]):_})
    3. {top3_sl[2][0]} ({round(top3_sl[2][1]):_})

    Топ 3 по RP:
    1. {top3_rp[0][0]} ({round(top3_rp[0][1]):_})
    2. {top3_rp[1][0]} ({round(top3_rp[1][1]):_})
    3. {top3_rp[2][0]} ({round(top3_rp[2][1]):_})

    Топ 3 по MP:
    1. {top3_mp[0][0]} ({round(top3_mp[0][1]):_})
    2. {top3_mp[1][0]} ({round(top3_mp[1][1]):_})
    3. {top3_mp[2][0]} ({round(top3_mp[2][1]):_})

    Топы по:
    Боям - {name_max_battle_count} ({value_max_battle_count} боев)
    КД - {name_max_kd} ({value_max_kd})
    Полезности - {name_max_objectives} ({round(value_max_objectives)} килов, критов, ассистов, баз)
    Выживаемости - {name_max_survivability} ({value_max_survivability})


    """.replace("_", " "))

    return {
        'top3_sl_name_1': top3_sl[0][0],
        'top3_sl_value_1': round(top3_sl[0][1]),
        'top3_sl_name_2': top3_sl[1][0],
        'top3_sl_value_2': round(top3_sl[1][1]),
        'top3_sl_name_3': top3_sl[2][0],
        'top3_sl_value_3': round(top3_sl[2][1]),
        'top3_rp_name_1': top3_rp[0][0],
        'top3_rp_value_1': round(top3_rp[0][1]),
        'top3_rp_name_2': top3_rp[1][0],
        'top3_rp_value_2': round(top3_rp[1][1]),
        'top3_rp_name_3': top3_rp[2][0],
        'top3_rp_value_3': round(top3_rp[2][1]),
        'top3_mp_name_1': top3_mp[0][0],
        'top3_mp_value_1': round(top3_mp[0][1]),
        'top3_mp_name_2': top3_mp[1][0],
        'top3_mp_value_2': round(top3_mp[1][1]),
        'top3_mp_name_3': top3_mp[2][0],
        'top3_mp_value_3': round(top3_mp[2][1]),
        'name_max_battle_count': name_max_battle_count,
        'value_max_battle_count': value_max_battle_count,
        'name_max_kd': name_max_kd,
        'value_max_kd': value_max_kd,
        'name_max_objectives': name_max_objectives,
        'value_max_objectives': round(value_max_objectives),
        'name_max_survivability': name_max_survivability,
        'value_max_survivability': value_max_survivability,
        'battle_count': battle_count
    }

In [60]:
data = generate_data_for_start_page(xlsx_path)


    455 боев
    Топ 3 по SL:
    1. ИС-6 (29 288)
    2. T58 (24 971)
    3. Magach 3 (ERA) (22 569)

    Топ 3 по RP:
    1. Magach 3 (ERA) (4 669)
    2. ИС-6 (4 523)
    3. T58 (4 313)

    Топ 3 по MP:
    1. M-51 (1 150)
    2. Strv m/41 S-I (938)
    3. ИС-6 (821)

    Топы по:
    Боям - T58 (60 боев)
    КД - F-84B-26 (inf)
    Полезности - M-51 (9 килов, критов, ассистов, баз)
    Выживаемости - F8F-1B (1.0)


    


In [61]:
class StartPageFrame(tk.Frame):
    def __init__(self, parent, data_dict, *args, **kwargs):
        super().__init__(parent, *args, **kwargs)
        self.data_dict = data_dict or {}
        self.widgets = {} # Для хранения ссылок на виджеты (опционально, для динамического обновления)
        self.create_widgets()

    def create_widgets(self):
        """Создает и размещает виджеты на фрейме."""
        # Очищаем фрейм на случай повторного использования
        for widget in self.winfo_children():
            widget.destroy()

        # --- Стили ---
        # Можно настроить шрифты и цвета
        header_font = ("Segoe UI", 14, "bold")
        section_font = ("Segoe UI", 12, "bold")
        value_font = ("Consolas", 10) # Consolas для чисел
        small_font = ("Segoe UI", 10)

        # --- Заголовок ---
        header_label = tk.Label(self, text=f"Топы по предыдущим боям ({self.data_dict['battle_count']})", font=header_font, pady=10)
        header_label.grid(row=0, column=0, columnspan=3, sticky="ew", padx=10)

        # --- Топ 3 по SL, RP, MP ---
        # Заголовки для топ-3
        tk.Label(self, text="🐱 По SL", font=section_font).grid(row=1, column=0, sticky="w", padx=(20, 5), pady=5)
        tk.Label(self, text="💡 По RP", font=section_font).grid(row=1, column=1, sticky="w", padx=5, pady=5)
        tk.Label(self, text="🌐 По MP", font=section_font).grid(row=1, column=2, sticky="w", padx=(5, 20), pady=5)

        # Данные топ-3
        # Используем имена и значения напрямую из data_dict
        for i in range(3):
            row = 2 + i
            # SL
            name_key = f'top3_sl_name_{i+1}'
            value_key = f'top3_sl_value_{i+1}'
            name = self.data_dict.get(name_key, "-")
            value = self.data_dict.get(value_key, "-")
            if isinstance(value, int):
                formatted_value = f"{value:_}".replace("_", " ")
            else:
                formatted_value = str(value)
            text_sl = f"{i+1}. {name} ({formatted_value})"
            tk.Label(self, text=text_sl, font=value_font, anchor="w").grid(row=row, column=0, sticky="w", padx=(30, 5))

            # RP
            name_key = f'top3_rp_name_{i+1}'
            value_key = f'top3_rp_value_{i+1}'
            name = self.data_dict.get(name_key, "-")
            value = self.data_dict.get(value_key, "-")
            if isinstance(value, int):
                formatted_value = f"{value:_}".replace("_", " ")
            else:
                formatted_value = str(value)
            text_rp = f"{i+1}. {name} ({formatted_value})"
            tk.Label(self, text=text_rp, font=value_font, anchor="w").grid(row=row, column=1, sticky="w", padx=5)

            # MP
            name_key = f'top3_mp_name_{i+1}'
            value_key = f'top3_mp_value_{i+1}'
            name = self.data_dict.get(name_key, "-")
            value = self.data_dict.get(value_key, "-")
            if isinstance(value, int):
                formatted_value = f"{value:_}".replace("_", " ")
            else:
                formatted_value = str(value)
            text_mp = f"{i+1}. {name} ({formatted_value})"
            tk.Label(self, text=text_mp, font=value_font, anchor="w").grid(row=row, column=2, sticky="w", padx=(5, 30))

        # --- Топ 1 по другим параметрам ---
        next_row = 5 # 2 (заголовки) + 3 (топ-3) 
        # Отступ
        tk.Label(self, text="", font=small_font, pady=5).grid(row=next_row, column=0, columnspan=3)
        next_row += 1

        # Заголовок "ТОП 1 по:"
        tk.Label(self, text="🌟 ТОП 1 по:", font=section_font, pady=5).grid(row=next_row, column=0, columnspan=3, sticky="w", padx=10)
        next_row += 1

        # Список параметров для отображения
        top1_params = [
            ('Боям', 'name_max_battle_count', 'value_max_battle_count'),
            ('КД', 'name_max_kd', 'value_max_kd'),
            ('Полезности', 'name_max_objectives', 'value_max_objectives'),
            ('Выживаемости', 'name_max_survivability', 'value_max_survivability')
        ]

        for i, (label_text, name_key, value_key) in enumerate(top1_params):
            name = self.data_dict.get(name_key, "-")
            val = self.data_dict.get(value_key, "-")
            # Форматируем значение, если это число
            if isinstance(val, (int, float)):
                if label_text == 'Выживаемости':
                     # Для выживаемости показываем проценты или доли
                    formatted_val = f"{val:.2f}" if isinstance(val, float) else str(val)
                else:
                    formatted_val = f"{val:_}".replace("_", " ")
            else:
                formatted_val = str(val)
            display_text = f"{label_text}: {name} ({formatted_val})"
            
            tk.Label(self, text=display_text, font=small_font, anchor="w").grid(row=next_row + i, column=0, columnspan=3, sticky="w", padx=30)

        # --- Конфигурация сетки ---
        # Растягиваем столбцы
        self.grid_columnconfigure(0, weight=1)
        self.grid_columnconfigure(1, weight=1)
        self.grid_columnconfigure(2, weight=1)
        # Отключаем автоматическое растягивание строк для предсказуемости
        # Можно включить, если нужно
        
    def update_data(self, new_data_dict):
        """Обновляет данные и пересоздает виджеты."""
        self.data_dict = new_data_dict
        self.create_widgets()

In [62]:
root = tk.Tk()
root.title("WT Stats Dashboard - Стартовая страница")
data_dict = generate_data_for_start_page(xlsx_path)
start_page_frame = StartPageFrame(parent=root, data_dict=data_dict)
start_page_frame.pack(fill="both", expand=True)
root.mainloop()


    455 боев
    Топ 3 по SL:
    1. ИС-6 (29 288)
    2. T58 (24 971)
    3. Magach 3 (ERA) (22 569)

    Топ 3 по RP:
    1. Magach 3 (ERA) (4 669)
    2. ИС-6 (4 523)
    3. T58 (4 313)

    Топ 3 по MP:
    1. M-51 (1 150)
    2. Strv m/41 S-I (938)
    3. ИС-6 (821)

    Топы по:
    Боям - T58 (60 боев)
    КД - F-84B-26 (inf)
    Полезности - M-51 (9 килов, критов, ассистов, баз)
    Выживаемости - F8F-1B (1.0)


    
